In [1]:
import numpy as np
import pandas as pd
import os, re
from pathlib import PureWindowsPath, Path
import json

USED ON THE WORK COMPUTER TO ACCESS THE ACTIVE FOLDER!

# Annotations

In [2]:

root = Path(r"Z:\_archived")

pattern = re.compile(r"DICOM-batch[1-3]-\w+")  # batch 1-3 and some annotator

# List all directories matching the pattern
dicom_folders = [p for p in root.iterdir() if p.is_dir() and pattern.fullmatch(p.name)]

for folder in dicom_folders:
    print(folder)

Z:\_archived\DICOM-batch1-kalina01
Z:\_archived\DICOM-batch1-kalina02
Z:\_archived\DICOM-batch1-kalina03
Z:\_archived\DICOM-batch1-kalina04
Z:\_archived\DICOM-batch1-kalina05
Z:\_archived\DICOM-batch1-kalina06
Z:\_archived\DICOM-batch1-kalina07
Z:\_archived\DICOM-batch1-kalina08
Z:\_archived\DICOM-batch1-kalina09
Z:\_archived\DICOM-batch1-kalina10
Z:\_archived\DICOM-batch1-kalina11
Z:\_archived\DICOM-batch1-kalina12
Z:\_archived\DICOM-batch1-kalina13
Z:\_archived\DICOM-batch1-kalina14
Z:\_archived\DICOM-batch1-kalina15
Z:\_archived\DICOM-batch1-kalina99
Z:\_archived\DICOM-batch2-david01
Z:\_archived\DICOM-batch2-giovanni01
Z:\_archived\DICOM-batch2-kalina99
Z:\_archived\DICOM-batch3-kalina01
Z:\_archived\DICOM-batch3-kalina99


In [3]:
selected = pd.read_csv('selected_scans.csv')
selected

,Original File,New File,Destination Folder
0,../DICOM-batch1\NKI-d23231-00-0063\20120416 CT...,NET_0000_0000.nii.gz,DICOM-batch1-kalina01
1,../DICOM-batch1\NKI-d23231-00-0070\20150821 CT...,NET_0001_0000.nii.gz,DICOM-batch1-kalina01
2,../DICOM-batch1\NKI-d23231-00-0036\20140218 CT...,NET_0002_0000.nii.gz,DICOM-batch1-kalina01
3,../DICOM-batch1\NKI-d23231-00-0071\20130924 CT...,NET_0003_0000.nii.gz,DICOM-batch1-kalina01
4,../DICOM-batch1\NKI-d23231-00-0077\20141222 CT...,NET_0004_0000.nii.gz,DICOM-batch1-kalina01
...,...,...,...
9007,../DICOM-batch3\NKI-d23231-00-0888\20220420 CT...,NET_5827_0000.nii.gz,DICOM-batch3-kalina99
9008,../DICOM-batch3\NKI-d23231-00-0888\20230424 CT...,NET_5828_0000.nii.gz,DICOM-batch3-kalina99
9009,../DICOM-batch3\NKI-d23231-00-0888\20230424 CT...,NET_5829_0000.nii.gz,DICOM-batch3-kalina99
9010,../DICOM-batch3\NKI-d23231-00-0888\20230801 CT...,NET_5830_0000.nii.gz,DICOM-batch3-kalina99


In [4]:

all_annot = pd.DataFrame()
first_csv = True  # Track if this is the first CSV being read

folder_path = Path(r"Z:\_archived")


pattern = re.compile(r"DICOM-batch[1-3]-\w+")  # batch 1-3 and some annotator

# List all directories matching the pattern
dicom_folders = [p for p in folder_path.iterdir() if p.is_dir() and pattern.fullmatch(p.name)]

for p in dicom_folders:
    if not p.is_dir():
        continue

    # List all CSVs in the folder
    csvs = [file for file in p.iterdir() if file.suffix == ".csv"]

    if not csvs:
        raise ValueError(f"No CSV files found in folder {p.name}.")

    # Pick the file according to your logic
    if len(csvs) > 1:
        csv2 = [f for f in csvs if f.name.endswith("_2.csv")]
        file_to_read = csv2[0] if csv2 else csvs[0]
    else:
        file_to_read = csvs[0]

    # Read CSV
    if first_csv:
        df = pd.read_csv(file_to_read, header=0)  # Use header from first CSV
        column_names = df.columns  # Store headers to reuse for next files
        first_csv = False
    else:
        df = pd.read_csv(file_to_read)
        df.columns = column_names  # Apply same column names as first CSV

    # Add folder name column
    df['folder'] = p.name

    # Append to master DataFrame
    all_annot = pd.concat([all_annot, df], ignore_index=True)

all_annot

,file,is_liver_imaged,contrast,phase_timing,no_lesions,ai_segmentation_quality,manual_segmentation_confidence,adjustment_segmentation_difficulty,high_signal_to_noise,metal_artifacts,patient_motion,other,comment,time-stamp,folder
0,NET_0000_0000.nii.gz,2,1,1.0,NaN,4.0,4.0,1.0,False,False,False,NaN,NaN,1.706176e+09,DICOM-batch1-kalina01
1,NET_0001_0000.nii.gz,NaN,NO_CONTRAST,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.706176e+09,DICOM-batch1-kalina01
2,NET_0002_0000.nii.gz,1,2,2.0,True,NaN,NaN,NaN,False,False,False,NaN,NaN,1.706176e+09,DICOM-batch1-kalina01
3,NET_0003_0000.nii.gz,1,1,2.0,True,NaN,NaN,NaN,True,False,False,NaN,NaN,1.706176e+09,DICOM-batch1-kalina01
4,NET_0004_0000.nii.gz,1,2,2.0,NaN,2.0,3.0,2.0,False,False,False,NaN,NaN,1.706177e+09,DICOM-batch1-kalina01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8901,NET_5823_0000.nii.gz,1,1,1.0,NaN,5.0,4.0,1.0,False,False,False,NaN,NaN,1.728478e+09,DICOM-batch3-kalina99
8902,NET_5825_0000.nii.gz,0,0,0.0,NaN,0.0,0.0,0.0,False,False,False,exclude - sagital,NaN,1.728478e+09,DICOM-batch3-kalina99
8903,NET_5827_0000.nii.gz,1,2,2.0,NaN,5.0,4.0,1.0,False,False,False,NaN,NaN,1.728478e+09,DICOM-batch3-kalina99
8904,NET_5828_0000.nii.gz,1,1,2.0,True,NaN,NaN,NaN,False,False,False,NaN,NaN,1.728478e+09,DICOM-batch3-kalina99


In [5]:
all_annot[all_annot[["file", "folder"]].duplicated(keep=False)].sort_values(by=["folder", "file"])  

,file,is_liver_imaged,contrast,phase_timing,no_lesions,ai_segmentation_quality,manual_segmentation_confidence,adjustment_segmentation_difficulty,high_signal_to_noise,metal_artifacts,patient_motion,other,comment,time-stamp,folder
55,NET_0055_0000.nii.gz,1,0,2.0,True,NaN,NaN,NaN,False,False,False,coronal reconstruction; EXCLUDE,NaN,1.706189e+09,DICOM-batch1-kalina01
56,NET_0055_0000.nii.gz,1,2,2.0,True,NaN,NaN,NaN,False,False,False,coronal reconstruction; EXCLUDE,NaN,1.706189e+09,DICOM-batch1-kalina01
81,NET_0021_0000.nii.gz,1.0,2,2.0,True,NaN,NaN,NaN,False,False,False,NaN,NaN,1.706261e+09,DICOM-batch1-kalina02
82,NET_0021_0000.nii.gz,1.0,2,2.0,True,NaN,NaN,NaN,False,False,False,NaN,NaN,1.706262e+09,DICOM-batch1-kalina02
877,NET_0184_0000.nii.gz,NaN,NO_CONTRAST,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.707745e+09,DICOM-batch1-kalina99
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8603,NET_5199_0000.nii.gz,NaN,NO_CONTRAST,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.728316e+09,DICOM-batch3-kalina99
5378,NET_5273_0000.nii.gz,1,2,2.0,True,NaN,NaN,NaN,False,False,False,NaN,NaN,1.721726e+09,DICOM-batch3-kalina99
5379,NET_5273_0000.nii.gz,1,2,3.0,True,NaN,NaN,NaN,False,False,False,NaN,NaN,1.721726e+09,DICOM-batch3-kalina99
3585,NET_5698_0000.nii.gz,1,2,2.0,True,NaN,NaN,NaN,False,False,False,NaN,NaN,1.720010e+09,DICOM-batch3-kalina99


In [6]:
idx = all_annot.groupby(["folder", "file"])["time-stamp"].idxmax()
annot_dedup = all_annot.loc[idx].sort_values(["folder", "file"])
annot_dedup

,file,is_liver_imaged,contrast,phase_timing,no_lesions,ai_segmentation_quality,manual_segmentation_confidence,adjustment_segmentation_difficulty,high_signal_to_noise,metal_artifacts,patient_motion,other,comment,time-stamp,folder
0,NET_0000_0000.nii.gz,2,1,1.0,NaN,4.0,4.0,1.0,False,False,False,NaN,NaN,1.706176e+09,DICOM-batch1-kalina01
1,NET_0001_0000.nii.gz,NaN,NO_CONTRAST,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.706176e+09,DICOM-batch1-kalina01
2,NET_0002_0000.nii.gz,1,2,2.0,True,NaN,NaN,NaN,False,False,False,NaN,NaN,1.706176e+09,DICOM-batch1-kalina01
3,NET_0003_0000.nii.gz,1,1,2.0,True,NaN,NaN,NaN,True,False,False,NaN,NaN,1.706176e+09,DICOM-batch1-kalina01
4,NET_0004_0000.nii.gz,1,2,2.0,NaN,2.0,3.0,2.0,False,False,False,NaN,NaN,1.706177e+09,DICOM-batch1-kalina01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8903,NET_5827_0000.nii.gz,1,2,2.0,NaN,5.0,4.0,1.0,False,False,False,NaN,NaN,1.728478e+09,DICOM-batch3-kalina99
8904,NET_5828_0000.nii.gz,1,1,2.0,True,NaN,NaN,NaN,False,False,False,NaN,NaN,1.728478e+09,DICOM-batch3-kalina99
4260,NET_5829_0000.nii.gz,1,2,2.0,NaN,5.0,3.0,1.0,False,False,False,NaN,NaN,1.720620e+09,DICOM-batch3-kalina99
3541,NET_5830_0000.nii.gz,1,1,2.0,True,NaN,NaN,NaN,False,False,False,NaN,NaN,1.720001e+09,DICOM-batch3-kalina99


In [7]:
annot_dedup.is_liver_imaged.value_counts()

is_liver_imaged
1           5606
0            381
2            348
1.0          286
NO_LIVER     118
2.0           31
Name: count, dtype: int64

In [8]:
annot_dedup_liver = annot_dedup[annot_dedup["is_liver_imaged"] != "NO_LIVER"]
print(annot_dedup_liver.shape, annot_dedup_liver.is_liver_imaged.isna().sum()) # Check for missing values in the filtered DataFrame

(8749, 15) 2097


In [9]:
annot_dedup_liver[annot_dedup_liver.is_liver_imaged.isna()]['contrast'].value_counts()

contrast
NO_CONTRAST    2097
Name: count, dtype: int64

In [10]:
display(annot_dedup_liver.contrast.value_counts(),
        annot_dedup_liver.is_liver_imaged.value_counts(),
        annot_dedup_liver.phase_timing.value_counts(),
        annot_dedup_liver.no_lesions.value_counts())

contrast
2              3708
1              2488
NO_CONTRAST    2097
0               441
3                15
Name: count, dtype: int64

is_liver_imaged
1      5606
0       381
2       348
1.0     286
2.0      31
Name: count, dtype: int64

phase_timing
2.0    5115
1.0     916
0.0     438
3.0     183
Name: count, dtype: int64

no_lesions
True    2151
Name: count, dtype: int64

In [11]:
annot_dedup_liver.to_csv("annotations.csv", index=False)

# DICOM

In [12]:
selected["Original File"].iloc[0]

'../DICOM-batch1\\NKI-d23231-00-0063\\20120416 CT abdomen carcinoid\\ABDOMEN A 1.0 FC02.nii.gz'

Change the .nii.gz with .json to extract the dicom header otherwise the path is the same
- Z:\DICOM\NKI-d23231-00-0063\20120416 CT abdomen carcinoid\ABDOMEN + C 1.0 FC02.json

In [13]:
base = Path(r"Z:\DICOM")

for i in selected["Original File"]:
    rel_path = Path(*PureWindowsPath(i).parts[2:])   # drop drive + top folder
    json_path = base / rel_path.with_suffix("").with_suffix(".json")

    print(json_path)
    break


Z:\DICOM\NKI-d23231-00-0063\20120416 CT abdomen carcinoid\ABDOMEN A 1.0 FC02.json


In [14]:
base = Path(r"Z:\DICOM")

for i in selected["Original File"]:
    rel = Path(*PureWindowsPath(i).parts[2:])
    json_path = base / rel.with_suffix("").with_suffix(".json")

    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    print(data)   # usually dict

    break

{'Modality': 'CT', 'ImagingFrequency': 0, 'Manufacturer': 'Toshiba', 'ManufacturersModelName': 'Aquilion', 'DeviceSerialNumber': '3936455', 'PatientPosition': 'FFS', 'SoftwareVersions': 'V4.51ER010', 'SeriesDescription': 'ABDOMEN A 1.0 FC02', 'ProtocolName': 'ABDOMEN A 1.0 FC02', 'ScanOptions': 'HELICAL_CT', 'ImageType': ['ORIGINAL', 'PRIMARY', 'AXIAL'], 'SeriesNumber': 7, 'AcquisitionTime': '11:10:31.700000', 'AcquisitionNumber': 6, 'ConvolutionKernel': 'FC02', 'ExposureTime': 0.5, 'XRayTubeCurrent': 150, 'XRayExposure': 75, 'ImageOrientationPatientDICOM': [1, 0, 0, 0, 1, 0], 'ConversionSoftware': 'dcm2niix', 'ConversionSoftwareVersion': 'v1.0.20230411'}


In [15]:
from pathlib import Path, PureWindowsPath
import json
import pandas as pd

base = Path(r"Z:\DICOM")
alt_base = Path(r"Z:\DICOM-batch-jacco")

records = []
missing = []
errors = []

for i in selected["Original File"]:
    row = {"Original File": i}   # <-- ALWAYS present

    rel = Path(*PureWindowsPath(i).parts[2:])
    json_path = base / rel.with_suffix("").with_suffix(".json")

    try:
        with open(json_path, "r", encoding="utf-8") as f:
            meta = json.load(f)

        for k, v in meta.items():
            if isinstance(v, (dict, list)):
                row[k] = json.dumps(v)
            else:
                row[k] = v

    except FileNotFoundError:
        json_path_alt = alt_base / rel.with_suffix("").with_suffix(".json")
        try:
            with open(json_path_alt, "r", encoding="utf-8") as f:
                meta = json.load(f)

            for k, v in meta.items():
                if isinstance(v, (dict, list)):
                    row[k] = json.dumps(v)
                else:
                    row[k] = v

        except FileNotFoundError:
            row["_json_missing"] = True
            missing.append(str(json_path))

    except json.JSONDecodeError as e:
        row["_json_error"] = True
        row["_json_error_msg"] = str(e)
        errors.append((str(json_path), str(e)))

    records.append(row)

# Build DataFrame
json_df = pd.DataFrame(records)

# Ensure column order
cols = ["Original File"] + [c for c in json_df.columns if c != "Original File"]
json_df = json_df[cols]

# Save
json_df.to_csv("dicom_metadata.csv", index=False)

print(f"Rows written: {len(json_df)}")
print(f"Missing JSONs: {len(missing)}")
print(f"Broken JSONs: {len(errors)}")



Rows written: 9012
Missing JSONs: 0
Broken JSONs: 0


In [18]:
json_df.head()

,Original File,Modality,ImagingFrequency,Manufacturer,ManufacturersModelName,DeviceSerialNumber,PatientPosition,SoftwareVersions,SeriesDescription,ProtocolName,...,ExposureTime,XRayTubeCurrent,XRayExposure,ImageOrientationPatientDICOM,ConversionSoftware,ConversionSoftwareVersion,BodyPartExamined,RawImage,SliceThickness,Units
0,../DICOM-batch1\NKI-d23231-00-0063\20120416 CT...,CT,0,Toshiba,Aquilion,3936455,FFS,V4.51ER010,ABDOMEN A 1.0 FC02,ABDOMEN A 1.0 FC02,...,0.5,150.0,75.0,"[1, 0, 0, 0, 1, 0]",dcm2niix,v1.0.20230411,NaN,NaN,NaN,NaN
1,../DICOM-batch1\NKI-d23231-00-0070\20150821 CT...,CT,0,Toshiba,Aquilion,3936455,FFS,V4.51ER010,LEVER -C 1.0 FC03,LEVER -C 1.0 FC03,...,0.5,76.0,38.0,"[1, 0, 0, 0, 1, 0]",dcm2niix,v1.0.20230411,NaN,NaN,NaN,NaN
2,../DICOM-batch1\NKI-d23231-00-0036\20140218 CT...,CT,0,Toshiba,Aquilion,3936455,FFS,V4.51ER010,ABDOMEN + C 1.0 FC01,ABDOMEN + C 1.0 FC01,...,0.5,75.0,37.0,"[1, 0, 0, 0, 1, 0]",dcm2niix,v1.0.20230411,NaN,NaN,NaN,NaN
3,../DICOM-batch1\NKI-d23231-00-0071\20130924 CT...,CT,0,Siemens,Sensation Open,662098617,FFS,syngo CT 2007S,Maag.abdomen 1.5 B30f,Maag.abdomen 1.5 B30f,...,0.5,168.0,84.0,"[1, 0, 0, 0, 1, 0]",dcm2niix,v1.0.20230411,ABDOMEN,NaN,NaN,NaN
4,../DICOM-batch1\NKI-d23231-00-0077\20141222 CT...,CT,0,Siemens,Sensation Open,662098617,FFS,syngo CT 2007S,Abdomen V 1.5 B25f,Abdomen V 1.5 B25f,...,0.5,119.0,74.0,"[1, 0, 0, 0, 1, 0]",dcm2niix,v1.0.20230411,ABDOMEN,NaN,NaN,NaN


# Extract .Nii Files and Segmentations

In [19]:
data = pd.read_csv("cleaned_data.csv")
data = data[data.file.notna()]
len(data[data.file.notna()])

8546

In [20]:
out_folder = Path(r"Z:\_archived")

data.insert(0, "NiiFile", Path(out_folder) / data['Destination Folder']/ data['New File'])
data.head()

,NiiFile,Original File,Destination Folder,New File,SubjectKeyRadiology,ExamDate,file,is_liver_imaged,contrast,phase_timing,...,XRayTubeCurrent,XRayExposure,ImageOrientationPatientDICOM,ConversionSoftware,ConversionSoftwareVersion,BodyPartExamined,RawImage,SliceThickness,AcquisitionTime_sec,DICOM_phase
0,Z:\_archived\DICOM-batch1-kalina01\NET_0000_00...,../DICOM-batch1\NKI-d23231-00-0063\20120416 CT...,DICOM-batch1-kalina01,NET_0000_0000.nii.gz,NKI-d23231-00-0063,2012-04-16,NET_0000_0000.nii.gz,Partially,Arterial,Too Early,...,150.0,75.0,"[1, 0, 0, 0, 1, 0]",dcm2niix,v1.0.20230411,ABDOMEN,NaN,1.0,40231.70000,Arterial
1,Z:\_archived\DICOM-batch1-kalina01\NET_0001_00...,../DICOM-batch1\NKI-d23231-00-0070\20150821 CT...,DICOM-batch1-kalina01,NET_0001_0000.nii.gz,NKI-d23231-00-0070,2015-08-21,NET_0001_0000.nii.gz,Yes,Non-contrast,Non-contrast,...,76.0,38.0,"[1, 0, 0, 0, 1, 0]",dcm2niix,v1.0.20230411,LEVER,NaN,1.0,30853.90000,Non-contrast
2,Z:\_archived\DICOM-batch1-kalina01\NET_0002_00...,../DICOM-batch1\NKI-d23231-00-0036\20140218 CT...,DICOM-batch1-kalina01,NET_0002_0000.nii.gz,NKI-d23231-00-0036,2014-02-18,NET_0002_0000.nii.gz,Yes,Portal,Just Right,...,75.0,37.0,"[1, 0, 0, 0, 1, 0]",dcm2niix,v1.0.20230411,ABDOMEN,NaN,1.0,32599.60000,NaN
3,Z:\_archived\DICOM-batch1-kalina01\NET_0003_00...,../DICOM-batch1\NKI-d23231-00-0071\20130924 CT...,DICOM-batch1-kalina01,NET_0003_0000.nii.gz,NKI-d23231-00-0071,2013-09-24,NET_0003_0000.nii.gz,Yes,Arterial,Just Right,...,168.0,84.0,"[1, 0, 0, 0, 1, 0]",dcm2niix,v1.0.20230411,ABDOMEN,NaN,1.5,45013.30692,NaN
4,Z:\_archived\DICOM-batch1-kalina01\NET_0004_00...,../DICOM-batch1\NKI-d23231-00-0077\20141222 CT...,DICOM-batch1-kalina01,NET_0004_0000.nii.gz,NKI-d23231-00-0077,2014-12-22,NET_0004_0000.nii.gz,Yes,Portal,Just Right,...,119.0,74.0,"[1, 0, 0, 0, 1, 0]",dcm2niix,v1.0.20230411,ABDOMEN,NaN,1.5,51587.10418,Portal


In [21]:
# Predefine root folder
out_folder = Path(r"Z:\_archived")

# Function to select the correct seg file path
def seg_path(row):
    folder = out_folder / row['Destination Folder']
    base_name = row['New File'].replace(".nii.gz", "")
    seg2 = folder / f"{base_name}.seg_2.nii.gz"
    seg1 = folder / f"{base_name}.seg.nii.gz"
    if seg2.exists():
        return seg2
    elif seg1.exists():
        return seg1
    else:
        return pd.NA

# Create column efficiently
data.insert(1, "SegNiiFile", [seg_path(row) for _, row in data.iterrows()])

data.head()

,NiiFile,SegNiiFile,Original File,Destination Folder,New File,SubjectKeyRadiology,ExamDate,file,is_liver_imaged,contrast,...,XRayTubeCurrent,XRayExposure,ImageOrientationPatientDICOM,ConversionSoftware,ConversionSoftwareVersion,BodyPartExamined,RawImage,SliceThickness,AcquisitionTime_sec,DICOM_phase
0,Z:\_archived\DICOM-batch1-kalina01\NET_0000_00...,Z:\_archived\DICOM-batch1-kalina01\NET_0000_00...,../DICOM-batch1\NKI-d23231-00-0063\20120416 CT...,DICOM-batch1-kalina01,NET_0000_0000.nii.gz,NKI-d23231-00-0063,2012-04-16,NET_0000_0000.nii.gz,Partially,Arterial,...,150.0,75.0,"[1, 0, 0, 0, 1, 0]",dcm2niix,v1.0.20230411,ABDOMEN,NaN,1.0,40231.70000,Arterial
1,Z:\_archived\DICOM-batch1-kalina01\NET_0001_00...,Z:\_archived\DICOM-batch1-kalina01\NET_0001_00...,../DICOM-batch1\NKI-d23231-00-0070\20150821 CT...,DICOM-batch1-kalina01,NET_0001_0000.nii.gz,NKI-d23231-00-0070,2015-08-21,NET_0001_0000.nii.gz,Yes,Non-contrast,...,76.0,38.0,"[1, 0, 0, 0, 1, 0]",dcm2niix,v1.0.20230411,LEVER,NaN,1.0,30853.90000,Non-contrast
2,Z:\_archived\DICOM-batch1-kalina01\NET_0002_00...,Z:\_archived\DICOM-batch1-kalina01\NET_0002_00...,../DICOM-batch1\NKI-d23231-00-0036\20140218 CT...,DICOM-batch1-kalina01,NET_0002_0000.nii.gz,NKI-d23231-00-0036,2014-02-18,NET_0002_0000.nii.gz,Yes,Portal,...,75.0,37.0,"[1, 0, 0, 0, 1, 0]",dcm2niix,v1.0.20230411,ABDOMEN,NaN,1.0,32599.60000,NaN
3,Z:\_archived\DICOM-batch1-kalina01\NET_0003_00...,Z:\_archived\DICOM-batch1-kalina01\NET_0003_00...,../DICOM-batch1\NKI-d23231-00-0071\20130924 CT...,DICOM-batch1-kalina01,NET_0003_0000.nii.gz,NKI-d23231-00-0071,2013-09-24,NET_0003_0000.nii.gz,Yes,Arterial,...,168.0,84.0,"[1, 0, 0, 0, 1, 0]",dcm2niix,v1.0.20230411,ABDOMEN,NaN,1.5,45013.30692,NaN
4,Z:\_archived\DICOM-batch1-kalina01\NET_0004_00...,Z:\_archived\DICOM-batch1-kalina01\NET_0004_00...,../DICOM-batch1\NKI-d23231-00-0077\20141222 CT...,DICOM-batch1-kalina01,NET_0004_0000.nii.gz,NKI-d23231-00-0077,2014-12-22,NET_0004_0000.nii.gz,Yes,Portal,...,119.0,74.0,"[1, 0, 0, 0, 1, 0]",dcm2niix,v1.0.20230411,ABDOMEN,NaN,1.5,51587.10418,Portal


In [22]:
print(data.NiiFile.iloc[0], data.SegNiiFile.iloc[0])

Z:\_archived\DICOM-batch1-kalina01\NET_0000_0000.nii.gz Z:\_archived\DICOM-batch1-kalina01\NET_0000_0000.seg_2.nii.gz


In [23]:
processed = pd.read_csv(r"Z:\active_Laura\dataset-registration-artinet.csv")
len(processed[processed.NiiFile.notna()])

5572

In [24]:
data["ExamDate"] = pd.to_datetime(data["ExamDate"], errors="coerce")
processed["ExamDate"] = pd.to_datetime(processed["ExamDate"], errors="coerce")


In [25]:
def create_key(data, col, fallback_label='unknown'):
    file_part = data['file'].astype('string').str.split('_').str[1]
    file_part = file_part.fillna(fallback_label) + '.nii.gz'   # pick your fallback label

    data[col] = (
        data['SubjectKeyRadiology'].astype('string')
        + "_"
        + data['Original File'].astype('string').apply(lambda x: PureWindowsPath(x).parts[3].split(" ")[0])
        + "_"
        + data['Destination Folder'].astype('string').str.split('-').str[-1]
        + "_"
        + file_part
    )


create_key(data, 'MatchKey')
processed['MatchKey'] = processed.NiiFile.apply(lambda x: PureWindowsPath(x).parts[-1])

processed.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 43 columns):
 #   Column                              Non-Null Count  Dtype         
---  ------                              --------------  -----         
 0   SubjectKeyRadiology                 5572 non-null   object        
 1   ExamDate                            5572 non-null   datetime64[ns]
 2   SegmentationBatch                   5572 non-null   object        
 3   NiiFile                             5572 non-null   object        
 4   SegNiiFile                          5572 non-null   object        
 5   file                                5572 non-null   object        
 6   is_liver_imaged                     5572 non-null   object        
 7   contrast                            5454 non-null   object        
 8   phase_timing                        5454 non-null   object        
 9   is_lesionfree                       5572 non-null   object        
 10  ai_segmentation_quality 

In [26]:
data = data.merge(
    processed[["MatchKey", "NiiFile"]],
    on=["MatchKey"],
    how="left",
    suffixes=("", "_proc"),
    
)
print(len(data[data.SubjectKeyRadiology.notna()]), len(data[data.NiiFile_proc.notna()]))

8546 5441


In [27]:
data = data.rename(columns={"NiiFile_proc": "exist_on_server"})
not_on_server = data[data.exist_on_server.isna()]
len(not_on_server)

3105

In [28]:
from pathlib import Path
import shutil

in_folder = Path(r"Z:\active_Kristina\not_on_server")
in_folder.mkdir(parents=True, exist_ok=True)


for row in not_on_server.itertuples(index=False, name=None):
    nii_path = Path(row[0])  # NiiFile
    seg_path = Path(row[1])  # SegNiiFile`
    dst = row[3]  # Destination Folder
    match_key = row[-2]


    if not nii_path.exists():
        print(f"Missing NIfTI: {nii_path}")
        continue

    if not seg_path.exists():
        print(f"Missing SEG: {seg_path}")
        continue

    # ---- Copy NIfTI ----
    nii_dst = in_folder / match_key
    shutil.copy2(nii_path, nii_dst)

    # ---- Copy segmentation ----
    seg_dst = in_folder / match_key.replace(".nii.gz", "_seg.nii.gz")
    shutil.copy2(seg_path, seg_dst)

print("✅ All files copied with MatchKey-based names!")

KeyboardInterrupt: 

In [29]:
data.to_csv("cleaned_data_matchkey_added.csv", index=False)

In [30]:
data.tail(20)

,NiiFile,SegNiiFile,Original File,Destination Folder,New File,SubjectKeyRadiology,ExamDate,file,is_liver_imaged,contrast,...,ImageOrientationPatientDICOM,ConversionSoftware,ConversionSoftwareVersion,BodyPartExamined,RawImage,SliceThickness,AcquisitionTime_sec,DICOM_phase,MatchKey,exist_on_server
8526,Z:\_archived\DICOM-batch3-kalina99\NET_5808_00...,Z:\_archived\DICOM-batch3-kalina99\NET_5808_00...,../DICOM-batch3\NKI-d23231-00-0887\20210318 Re...,DICOM-batch3-kalina99,NET_5808_0000.nii.gz,NKI-d23231-00-0887,2021-03-18,NET_5808_0000.nii.gz,Yes,Portal,...,"[1, 0, 0, 0, 1, 0]",dcm2niix,v1.0.20230411,CT THORAX EN ___,NaN,3.0,29400.19800,NaN,NKI-d23231-00-0887_20210318_kalina99_5808.nii.gz,\\nki.nl\res\RD CRC-data\_archive\IRBd23231-AM...
8527,Z:\_archived\DICOM-batch3-kalina99\NET_5811_00...,Z:\_archived\DICOM-batch3-kalina99\NET_5811_00...,../DICOM-batch3\NKI-d23231-00-0887\20210825 CT...,DICOM-batch3-kalina99,NET_5811_0000.nii.gz,NKI-d23231-00-0887,2021-08-25,NET_5811_0000.nii.gz,Yes,Arterial,...,"[1, 0, 0, 0, 1, 0]",dcm2niix,v1.0.20230411,ABDOMEN,NaN,1.5,55911.51069,Arterial,NKI-d23231-00-0887_20210825_kalina99_5811.nii.gz,\\nki.nl\res\RD CRC-data\_archive\IRBd23231-AM...
8528,Z:\_archived\DICOM-batch3-kalina99\NET_5812_00...,Z:\_archived\DICOM-batch3-kalina99\NET_5812_00...,../DICOM-batch3\NKI-d23231-00-0887\20210825 CT...,DICOM-batch3-kalina99,NET_5812_0000.nii.gz,NKI-d23231-00-0887,2021-08-25,NET_5812_0000.nii.gz,Yes,Portal,...,"[1, 0, 0, 0, 1, 0]",dcm2niix,v1.0.20230411,ABDOMEN,NaN,1.5,55943.93233,Portal,NKI-d23231-00-0887_20210825_kalina99_5812.nii.gz,\\nki.nl\res\RD CRC-data\_archive\IRBd23231-AM...
8529,Z:\_archived\DICOM-batch3-kalina99\NET_5813_00...,Z:\_archived\DICOM-batch3-kalina99\NET_5813_00...,../DICOM-batch3\NKI-d23231-00-0887\20220210 CT...,DICOM-batch3-kalina99,NET_5813_0000.nii.gz,NKI-d23231-00-0887,2022-02-10,NET_5813_0000.nii.gz,Yes,Portal,...,"[1, 0, 0, 0, 1, 0]",dcm2niix,v1.0.20230411,ABDOMEN,NaN,1.0,51448.80000,NaN,NKI-d23231-00-0887_20220210_kalina99_5813.nii.gz,NaN
8530,Z:\_archived\DICOM-batch3-kalina99\NET_5814_00...,Z:\_archived\DICOM-batch3-kalina99\NET_5814_00...,../DICOM-batch3\NKI-d23231-00-0887\20220415 CT...,DICOM-batch3-kalina99,NET_5814_0000.nii.gz,NKI-d23231-00-0887,2022-04-15,NET_5814_0000.nii.gz,Yes,Portal,...,"[1, 0, 0, 0, 1, 0]",dcm2niix,v1.0.20230411,ABDOMEN,NaN,1.0,52780.25000,NaN,NKI-d23231-00-0887_20220415_kalina99_5814.nii.gz,NaN
8531,Z:\_archived\DICOM-batch3-kalina99\NET_5815_00...,Z:\_archived\DICOM-batch3-kalina99\NET_5815_00...,../DICOM-batch3\NKI-d23231-00-0887\20220721 CT...,DICOM-batch3-kalina99,NET_5815_0000.nii.gz,NKI-d23231-00-0887,2022-07-21,NET_5815_0000.nii.gz,Yes,Arterial,...,"[1, 0, 0, 0, 1, 0]",dcm2niix,v1.0.20230411,ABDOMEN,NaN,1.5,49781.80799,Arterial,NKI-d23231-00-0887_20220721_kalina99_5815.nii.gz,NaN
8532,Z:\_archived\DICOM-batch3-kalina99\NET_5816_00...,Z:\_archived\DICOM-batch3-kalina99\NET_5816_00...,../DICOM-batch3\NKI-d23231-00-0887\20220721 CT...,DICOM-batch3-kalina99,NET_5816_0000.nii.gz,NKI-d23231-00-0887,2022-07-21,NET_5816_0000.nii.gz,Yes,Portal,...,"[1, 0, 0, 0, 1, 0]",dcm2niix,v1.0.20230411,ABDOMEN,NaN,1.5,49814.10341,Portal,NKI-d23231-00-0887_20220721_kalina99_5816.nii.gz,\\nki.nl\res\RD CRC-data\_archive\IRBd23231-AM...
8533,Z:\_archived\DICOM-batch3-kalina99\NET_5817_00...,Z:\_archived\DICOM-batch3-kalina99\NET_5817_00...,../DICOM-batch3\NKI-d23231-00-0887\20221028 CT...,DICOM-batch3-kalina99,NET_5817_0000.nii.gz,NKI-d23231-00-0887,2022-10-28,NET_5817_0000.nii.gz,Yes,Portal,...,"[1, 0, 0, 0, 1, 0]",dcm2niix,v1.0.20230411,ABDOMEN,NaN,1.0,57450.45000,NaN,NKI-d23231-00-0887_20221028_kalina99_5817.nii.gz,\\nki.nl\res\RD CRC-data\_archive\IRBd23231-AM...
8534,Z:\_archived\DICOM-batch3-kalina99\NET_5818_00...,Z:\_archived\DICOM-batch3-kalina99\NET_5818_00...,../DICOM-batch3\NKI-d23231-00-0887\20230524 CT...,DICOM-batch3-kalina99,NET_5818_0000.nii.gz,NKI-d23231-00-0887,2023-05-24,NET_5818_0000.nii.gz,Yes,Portal,...,"[1, 0, 0, 0, 1, 0]",dcm2niix,v1.0.2